### Notas para la estimación objetiva de la confianza

In [ ]:
# TODO: Estimar objetivamente la confianza de la extracción con IA
# confidence = 1.0

# confidence = 1.0

# if project_name is None:
#     confidence -= 0.2

# if promoter is None:
#     confidence -= 0.1

# if len(main_events) == 0:
#     confidence -= 0.3

# if len(locations) == 0:
#     confidence -= 0.1

# if project_name is None:
#     confidence -= 0.2

# if promoter is None:
#     confidence -= 0.1

# if len(main_events) == 0:
#     confidence -= 0.3

# if len(locations) == 0:
#     confidence -= 0.1

# df[
#     (df["relevance_confidence"] < 0.7)
#     | (df["extraction_confidence"] < 0.7)
# ]

In [ ]:
import json
import random
import re
import time
from datetime import date, datetime, timezone
from enum import Enum
from pathlib import Path
from typing import Any

import pandas as pd
from pydantic import BaseModel, Field
from pydantic_ai import Agent

from renewables_permitting.utils import (
    normalize_text_or_none,
    save_parquet,
    validate_required_columns,
)

BASE_URL = "https://www.boe.es/datosabiertos/api/boe/sumario"

# PROJECT_ROOT = Path(__file__).resolve().parents[2]  # fuera del notebook
PROJECT_ROOT = Path.cwd().parent  # dentro del notebook

DATA_DIR = PROJECT_ROOT / "data"

BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"

# BRONZE
BOE_DOCS_XML_DIR = BRONZE_DIR / "boe_docs_xml"


# SILVER
BOE_CANDIDATES_PATH = SILVER_DIR / "boe_candidates" / "boe_candidates_normalized.parquet"

BOE_CANDIDATES_DOCS_TEXT_PATH = SILVER_DIR / "boe_candidates_docs_text" / "boe_candidates_docs_text.parquet"

DIM_MUNICIPALITIES_PATH = SILVER_DIR / "dimensions" / "dim_municipalities.parquet"

SILVER_BOE_AI_DIR = SILVER_DIR / "boe_ai"

BOE_AI_EXTRACTIONS_PATH = SILVER_BOE_AI_DIR / "boe_ai_extractions.parquet"
LIFECYCLE_EVENTS_PATH = SILVER_BOE_AI_DIR / "lifecycle_events.parquet"
ADMINISTRATIVE_ACTIONS_PATH = SILVER_BOE_AI_DIR / "administrative_actions.parquet"
ASSET_MENTIONS_PATH = SILVER_BOE_AI_DIR / "asset_mentions.parquet"
ASSET_TECHNOLOGIES_PATH = SILVER_BOE_AI_DIR / "asset_technologies.parquet"
ASSET_PARTICIPANTS_PATH = SILVER_BOE_AI_DIR / "asset_participants.parquet"
#ASSET_LOCATIONS_PATH = SILVER_BOE_AI_DIR / "asset_locations.parquet"
ASSET_ALIASES_PATH = SILVER_BOE_AI_DIR / "asset_aliases.parquet"
ASSET_RELATION_MENTIONS_PATH = SILVER_BOE_AI_DIR / "asset_relation_mentions.parquet"

ASSET_LOCATIONS_ENRICHED_PATH = SILVER_DIR / "boe_ai_deterministic_enrichment" / "asset_locations_enriched.parquet"


# GOLD
PROJECT_GROUPS_PATH = GOLD_DIR / "project_groups.parquet"
PROJECT_ASSETS_PATH = GOLD_DIR / "project_assets.parquet"
PROJECT_TIMELINE_PATH = GOLD_DIR / "project_timeline.parquet"
PROJECT_STATUS_PATH = GOLD_DIR / "project_status.parquet"

## 1. Construir asset_mentions_enriched

In [4]:
# 1. Cargar tablas flatten
lifecycle_events = pd.read_parquet(LIFECYCLE_EVENTS_PATH)
administrative_actions = pd.read_parquet(ADMINISTRATIVE_ACTIONS_PATH)
asset_mentions = pd.read_parquet(ASSET_MENTIONS_PATH)
asset_technologies = pd.read_parquet(ASSET_TECHNOLOGIES_PATH)
asset_participants = pd.read_parquet(ASSET_PARTICIPANTS_PATH)
asset_locations = pd.read_parquet(ASSET_LOCATIONS_ENRICHED_PATH)
asset_aliases = pd.read_parquet(ASSET_ALIASES_PATH)
asset_relation_mentions = pd.read_parquet(ASSET_RELATION_MENTIONS_PATH)

In [5]:
def aggregate_list(df, group_col, value_col):
    return (
        df.dropna(subset=[value_col])
        .groupby(group_col)[value_col]
        .apply(lambda s: sorted(set(s.astype(str))))
        .reset_index()
    )

In [6]:
asset_mentions_enriched = asset_mentions.copy()

asset_mentions_enriched = asset_mentions_enriched.merge(
    aggregate_list(asset_technologies, "asset_mention_id", "technology_type"),
    on="asset_mention_id",
    how="left",
)

asset_mentions_enriched = asset_mentions_enriched.merge(
    aggregate_list(asset_locations, "asset_mention_id", "municipality"),
    on="asset_mention_id",
    how="left",
)

asset_mentions_enriched = asset_mentions_enriched.merge(
    aggregate_list(asset_locations, "asset_mention_id", "province"),
    on="asset_mention_id",
    how="left",
)

asset_mentions_enriched = asset_mentions_enriched.merge(
    aggregate_list(asset_locations, "asset_mention_id", "autonomous_community"),
    on="asset_mention_id",
    how="left",
)

asset_mentions_enriched = asset_mentions_enriched.merge(
    aggregate_list(asset_participants, "asset_mention_id", "participant_name"),
    on="asset_mention_id",
    how="left",
)

asset_mentions_enriched = asset_mentions_enriched.merge(
    aggregate_list(asset_aliases, "asset_mention_id", "alias"),
    on="asset_mention_id",
    how="left",
)

In [10]:
display(asset_mentions_enriched.loc[asset_mentions_enriched["asset_mention_id"].str.contains("BOE-B-2021-32560")])

,asset_mention_id,event_id,identificador_boe,fecha_publicacion,local_asset_id,asset_name,asset_name_norm,role_in_event,status_in_document,evidence,technology_type,municipality,province,autonomous_community,participant_name,alias
0,BOE-B-2021-32560_event_1_asset_1,BOE-B-2021-32560_event_1,BOE-B-2021-32560,2021-07-07,asset_1,Parque eólico Badulaque,parque eolico badulaque,main_asset,under_permitting,Denominación: Parque eólico Badulaque de 90 MW...,[eolica],"[Cedeira, Cerdido, Moeche, Pontes de García Ro...","[Coruña, A]",[Galicia],"[ENEL GREEN POWER ESPAÑA, S.L.]",[Parque eólico Badulaque de 90 MW]
1,BOE-B-2021-32560_event_1_asset_2,BOE-B-2021-32560_event_1,BOE-B-2021-32560,2021-07-07,asset_2,SET 33/220 kV Badulaque,set 33 220 kv badulaque,associated_asset,planned,2. SET 33/220 kV Badulaque.,NaN,NaN,NaN,NaN,NaN,NaN
2,BOE-B-2021-32560_event_1_asset_3,BOE-B-2021-32560_event_1,BOE-B-2021-32560,2021-07-07,asset_3,Subestación Colectora SET Colectora 33/220/400...,subestacion colectora set colectora 33 220 400...,associated_asset,planned,3. Subestación Colectora SET Colectora 33/220/...,NaN,NaN,NaN,NaN,NaN,NaN
3,BOE-B-2021-32560_event_1_asset_4,BOE-B-2021-32560_event_1,BOE-B-2021-32560,2021-07-07,asset_4,Línea subterránea de 400 kV entre la SE Colect...,linea subterranea de 400 kv entre la se colect...,associated_asset,planned,5. Línea subterránea de 400 kV entre la SE Col...,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
def print_boe_audit(identificador_boe: str):
    events = lifecycle_events[
        lifecycle_events["identificador_boe"] == identificador_boe
    ]

    for _, event in events.iterrows():
        event_id = event["event_id"]

        print(f"\n{identificador_boe}")
        print("-" * len(identificador_boe))
        print(f"EVENTO: {event_id}")
        print(f"Tipo: {event.get('event_type')}")
        print(f"Etapa: {event.get('administrative_stage')}")
        print(f"Decisión: {event.get('administrative_decision')}")
        print(f"Resumen: {event.get('event_summary')}")
        print(f"Evidencia: {event.get('evidence')}")

        print("\nACTIVOS")
        assets = asset_mentions_enriched[
            asset_mentions_enriched["event_id"] == event_id
        ]

        for _, asset in assets.iterrows():
            print(f"- {asset.get('asset_name')}")
            print(f"  Rol: {asset.get('role_in_event')}")
            print(f"  Estado: {asset.get('status_in_document')}")
            print(f"  Tecnologías: {asset.get('technology_type')}")
            print(f"  Municipios: {asset.get('municipality')}")
            print(f"  Participantes: {asset.get('participant_name')}")
            print(f"  Alias: {asset.get('alias')}")

        print("\nACCIONES ADMINISTRATIVAS")
        actions = administrative_actions[
            administrative_actions["event_id"] == event_id
        ]

        for _, action in actions.iterrows():
            print(
                f"- {action.get('administrative_stage')} | "
                f"{action.get('administrative_decision')} | "
                f"{action.get('evidence')}"
            )

In [9]:
print_boe_audit("BOE-B-2021-32560")
# print_boe_audit("BOE-A-2023-10306")
# print_boe_audit("BOE-A-2025-18285")
# print_boe_audit("BOE-B-2026-3596")


BOE-B-2021-32560
----------------
EVENTO: BOE-B-2021-32560_event_1
Tipo: new_project
Etapa: None
Decisión: None
Resumen: Sometimiento a información pública del Estudio de Impacto Ambiental y la solicitud de Autorización Administrativa Previa para el Parque Eólico Badulaque de 90 MW y su infraestructura de evacuación.
Evidencia: Anuncio del Área de Industria y Energía de la Delegación del Gobierno en Galicia por el que se somete a información pública el Estudio de Impacto Ambiental y la solicitud de Autorización Administrativa Previa del Parque Eólico Badulaque de 90 MW y su infraestructura de evacuación en la provincia de A Coruña.

ACTIVOS
- Parque eólico Badulaque
  Rol: main_asset
  Estado: under_permitting
  Tecnologías: ['eolica']
  Municipios: ['Cedeira', 'Cerdido', 'Moeche', 'Pontes de García Rodríguez, As', 'Somozas, As', 'Valdoviño']
  Participantes: ['ENEL GREEN POWER ESPAÑA, S.L.']
  Alias: ['Parque eólico Badulaque de 90 MW']
- SET 33/220 kV Badulaque
  Rol: associated_ass

## 2. Construir event_audit_view

## 3. Mostrar fichas legibles por BOE / event_id